# Deep learning: from pixels to digit probabilities

## Introduction

How does a neural network turn an image into a prediction? This notebook revisits *Deep Learning* through a small, visible example: recognizing handwritten digits from 8×8 pixel images.

The worked examples connect image features to a network with two hidden layers, trace its learning over epochs, select a checkpoint using validation data, and reconstruct its forward calculation. This compact network makes the lecture's multilayer principles visible.

**Outline:** [Key Concepts](#key-concepts) · [Perceptrons and neural networks](#perceptrons-and-neural-networks) · [Neural network training](#neural-network-training) · [Network outputs and applications](#network-outputs-and-applications) · [Summary](#summary) · [Tasks](#tasks)

**Lecture connection:** slides 13–25 (activation functions and layers), 27–29 (gradient-based learning), 35–40 (task-specific outputs), and 42–45 (architecture and early stopping).

We'll use **pandas** to load the digit data, **NumPy** to follow the activation calculations, and **Matplotlib** to display images and learning curves. **Scikit-learn** provides the network and evaluation tools. `deepcopy` from Python's standard library preserves a fitted checkpoint while training continues.

In [ ]:
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, log_loss, ConfusionMatrixDisplay

# Reuse one seed for every random operation in this notebook.
SEED = 67


## Key Concepts

An image is represented by a feature vector $\mathbf{x}_i\in\mathbb{R}^{64}$, one value per pixel. Here $n$ counts images and $d=64$ counts features. For this multiclass application, $y_i\in\{0,\ldots,9\}$ records the digit; the target identifies an image’s digit class.

A hidden layer transforms a weighted sum through a nonlinear activation. Using local network notation, let $\mathbf{h}_i^{(0)}=\mathbf{x}_i$. For hidden layer $\ell$,

$$\mathbf{h}_i^{(\ell)}=\operatorname{ReLU}\!\left(\widehat{\mathbf{b}}^{(\ell)}+
\widehat{\mathbf{W}}^{(\ell)}\mathbf{h}_i^{(\ell-1)}\right),\qquad
\operatorname{ReLU}(z)=\max(0,z).$$

The hatted matrix and vector contain fitted connection weights and biases. ReLU acts on each component. Nonlinear activations allow the network to represent nonlinear mappings.

The output layer produces a vector of *logits*, $\mathbf{u}_i$. For digit label $r\in\{0,\ldots,9\}$, *softmax* gives

$$\widehat{p}_{ir}=\frac{\exp(u_{ir})}{\sum_{q=0}^{9}\exp(u_{iq})}.$$

The probabilities sum to one, and the predicted digit has the largest probability. A binary task needs one independent probability; regression typically uses an unconstrained numerical output.

An *epoch* is one pass through the training observations. Backpropagation calculates gradients, and a gradient-based optimizer updates weights. Training data supply the gradients, and validation data guide stopping and model choices. Test data remain reserved for the final assessment.

## Perceptrons and neural networks

### Inspect the image data

The next cell loads a frozen export of **scikit-learn**'s bundled *Optical Recognition of Handwritten Digits* dataset: 1,797 images, 64 pixel features and 10 digit labels. Pixel intensities are integers from 0 to 16. Columns `pixel_0` to `pixel_63` follow image rows from top left to bottom right.

The original data are attributed to E. Alpaydin and C. Kaynak (1998). The supplied rows come from the source dataset's original test subset; this notebook divides those rows into training, validation and test samples for the demonstration. Dataset provenance is recorded in `../data/handwritten_digits_source.txt`. The notebook loads the supplied local CSV.

In [ ]:
digits = pd.read_csv('../data/handwritten_digits.csv')
X = digits.drop(columns='digit') / 16.0
y = digits['digit']
print(f'{len(X)} images; {X.shape[1]} features; labels {sorted(y.unique().tolist())}')
fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for label, ax in enumerate(axes.ravel()):
    image = X.loc[y == label].iloc[0].to_numpy().reshape(8, 8)
    ax.imshow(image, cmap='gray_r', vmin=0, vmax=1)
    ax.set_title(f'Digit {label}')
    ax.axis('off')
plt.tight_layout()
plt.show()

Each row represents one image. `.reshape(8, 8)` restores the spatial arrangement for plotting. The fully connected network below receives the flattened vector of 64 pixel values.

Dividing by the known intensity maximum, 16, puts inputs on a 0–1 scale. This normalization uses the fixed measurement range of the pixel values.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED
)
print(f'Training images: {len(X_train)}; validation: {len(X_valid)}; test: {len(X_test)}')

### A neuron and its activation

![Deep Learning slide 15: hidden units transform weighted inputs before passing them to the next layer.](../resources/5_hidden_layer.png)

ReLU keeps positive inputs and sets negative inputs to zero. The plot below compares it with the identity mapping, making the nonlinearity visible. The horizontal axis shows the activation argument.

In [ ]:
activation_input = np.linspace(-3, 3, 200)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(activation_input, np.maximum(0, activation_input), label='ReLU', linewidth=2)
ax.plot(activation_input, activation_input, linestyle='--', label='Identity')
ax.set(xlabel='Activation input (unitless)', ylabel='Activation output (unitless)', title='A nonlinear hidden-layer activation')
ax.legend()
plt.tight_layout()
plt.show()

### Specify the architecture

<p style="color: red;"><strong>Disclaimer: Please note that the following interpretation assumes you executed the original code. Changes can invalidate our interpretation.</strong></p>

`MLPClassifier` is a multilayer perceptron classifier. `hidden_layer_sizes=(32, 16)` specifies **two hidden layers**: the architecture is 64 → 32 → 16 → 10. `activation='relu'` applies to the hidden layers; multiclass classification uses softmax at the output automatically.

`solver='adam'` selects an adaptive gradient-based optimizer. `learning_rate_init` controls the initial step scale, and `alpha` controls L2 regularization. These settings define the network used in the demonstration. We will use `partial_fit` for one training epoch at a time, making stopping explicit.

In [ ]:
network = MLPClassifier(
    hidden_layer_sizes=(32, 16), activation='relu', solver='adam',
    learning_rate_init=0.003, alpha=0.001, batch_size=64, random_state=SEED
)

## Neural network training

![Deep Learning slide 28: repeatedly calculate outputs, compute loss and update connection weights.](../resources/5_network_training.png)

### Follow learning and preserve the best checkpoint

The first `partial_fit` call receives `classes=np.arange(10)` so the model knows all possible labels. Later calls continue updating the existing weights and optimizer state.

The loop measures multiclass log-loss on training and validation images after each epoch. Lower is better. `patience=12` allows twelve consecutive epochs without a new best validation loss before stopping; `max_epochs` is a separate upper limit. Whenever validation loss reaches a new minimum, the loop saves a copy of the network; subsequent predictions use the best saved checkpoint.

This is a short transparent training loop. Training images supply weight updates, and validation images supply the stopping criterion. Test images remain reserved for the final assessment.

In [ ]:
max_epochs = 150
patience = 12
best_validation_loss = np.inf
epochs_without_improvement = 0
history = []

for epoch in range(1, max_epochs + 1):
    network.partial_fit(X_train, y_train, classes=np.arange(10))
    training_loss = log_loss(y_train, network.predict_proba(X_train), labels=network.classes_)
    validation_loss = log_loss(y_valid, network.predict_proba(X_valid), labels=network.classes_)
    history.append({'epoch': epoch, 'training_loss': training_loss, 'validation_loss': validation_loss})
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_epoch = epoch
        best_network = deepcopy(network)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    if epochs_without_improvement >= patience:
        break

learning_history = pd.DataFrame(history).set_index('epoch')
print(f'Training ended after {epoch} epochs; selected checkpoint: epoch {best_epoch}')
learning_history.tail().round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
learning_history.plot(ax=ax)
ax.axvline(best_epoch, linestyle='--', color='black', label='Selected checkpoint')
ax.set(xlabel='Epoch', ylabel='Multiclass log-loss', title='Learning from training images; monitoring validation images')
ax.legend(['Training', 'Validation', 'Selected checkpoint'])
plt.tight_layout()
plt.show()

<p style="color: red;"><strong>Disclaimer: Please note that the following interpretation assumes you executed the original code. Changes can invalidate our interpretation.</strong></p>

Inspect where validation improvement slows compared with training improvement. Validation loss guides the stopping decision. This run may stop by patience or by the epoch limit; either way, the selected checkpoint is the one with the lowest observed validation loss. The reserved test sample assesses the selected checkpoint.

![Deep Learning slide 44: validation evidence can identify when to stop training.](../resources/5_early_stopping.png)

The blue curve in the slide is labeled “Testing”; it represents the **validation** role used for stopping decisions in this example. The final test set must stay separate.

Other ways to control a network include its width, depth and regularization. The lecture also introduces dropout. Our `MLPClassifier` example uses L2 regularization and validation-based checkpoint selection.

## Network outputs and applications

### Inspect the fitted arrays

`coefs_` contains one connection matrix per layer transition; `intercepts_` contains the corresponding bias arrays. Scikit-learn stores a transition as **inputs × outputs**, so a row of observations uses `row @ weights + bias`. This is the transpose of the column-vector convention in Key Concepts.

The next cell counts weights and biases, showing how the architecture determines the number of trainable parameters.

In [ ]:
layer_shapes = pd.DataFrame({
    'transition': ['Input → hidden 1', 'Hidden 1 → hidden 2', 'Hidden 2 → output'],
    'weight_shape': [weights.shape for weights in best_network.coefs_],
    'bias_shape': [bias.shape for bias in best_network.intercepts_]
})
print(layer_shapes.to_string(index=False))
parameter_count = sum(weights.size for weights in best_network.coefs_) + sum(bias.size for bias in best_network.intercepts_)
print(f'Trainable weights and biases: {parameter_count}')

### Reconstruct one forward pass

![Deep Learning slide 40: softmax transforms raw output scores into class probabilities that sum to one.](../resources/5_softmax.png)

For one validation image, the next cell computes both ReLU layers, the output logits and softmax. The `@` operator performs matrix multiplication. NumPy adds the bias elementwise after each multiplication.

Subtracting the largest logit before exponentiation avoids unnecessarily large numbers while preserving the probabilities: the common exponential factor cancels between numerator and denominator.

In [ ]:
image_row = X_valid.iloc[[0]]
hidden_1 = np.maximum(0, image_row.to_numpy() @ best_network.coefs_[0] + best_network.intercepts_[0])
hidden_2 = np.maximum(0, hidden_1 @ best_network.coefs_[1] + best_network.intercepts_[1])
logits = hidden_2 @ best_network.coefs_[2] + best_network.intercepts_[2]
exponentials = np.exp(logits - logits.max(axis=1, keepdims=True))
manual_probability = exponentials / exponentials.sum(axis=1, keepdims=True)
sklearn_probability = best_network.predict_proba(image_row)
pd.DataFrame({
    'digit': best_network.classes_,
    'manual_probability': manual_probability[0],
    'sklearn_probability': sklearn_probability[0]
}).round(6)

In [ ]:
print(f'Sum of manual probabilities: {manual_probability.sum():.6f}')
print(f'Largest absolute difference: {np.abs(manual_probability - sklearn_probability).max():.2e}')
print(f'Predicted digit: {best_network.predict(image_row)[0]}; observed digit: {y_valid.iloc[0]}')

<p style="color: red;"><strong>Disclaimer: Please note that the following interpretation assumes you executed the original code. Changes can invalidate our interpretation.</strong></p>

The two calculations agree. `.predict_proba` returns ten columns in `classes_` order, while `.predict` returns the label with the largest probability. A probability vector and a class label carry different amounts of information.

The lecture's business settings use the same principle: one numeric output for a resale-price regression, a sigmoid probability for default, or multiple softmax outputs for mutually exclusive rating grades. Multiple independent binary labels require a different output interpretation from these mutually exclusive digit classes.

### Evaluate the frozen checkpoint

The selected network remains at its saved checkpoint. The next cells compare its test accuracy with a majority-class baseline fitted only on the training labels and show its test confusion matrix. The test images enter only at this final assessment, after training and checkpoint selection are complete.

In [ ]:
test_prediction = best_network.predict(X_test)
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
pd.Series({
    'Selected network test accuracy': accuracy_score(y_test, test_prediction),
    'Majority-class test accuracy': baseline.score(X_test, y_test)
}).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_test, test_prediction, labels=np.arange(10),
                                       cmap='Blues', colorbar=False, ax=ax)
ax.set_title('Handwritten digits: final test predictions')
plt.tight_layout()
plt.show()

Rows are observed digits and columns are predictions. Identify an off-diagonal cell and explain its direction: for example, a row-3, column-8 entry counts threes predicted as eights.

The reported accuracy describes these small, normalized images. Photographs, new writing styles and other collection conditions would require separate evaluation. Inspecting examples helps you understand what aggregate metrics conceal.

## Summary

- Hidden layers combine weighted sums and nonlinear activations; architecture determines array shapes and parameter counts.
- Gradient-based training adjusts weights over epochs; validation data guide checkpoint selection.
- A forward pass is a sequence of matrix multiplications, bias additions and activations.
- Multiclass softmax probabilities sum to one; the highest-probability label is the class prediction.
- The reserved test set assesses the selected checkpoint after the training choices have been fixed.

Here's a quick reference for the new Python tools introduced in this notebook:

| Python pattern | What it helps you do |
|---|---|
| `ax.imshow` | Display an image from its pixel intensities |
| `MLPClassifier(hidden_layer_sizes=(32, 16))` | Specify a network with two hidden layers |
| `model.partial_fit(..., classes=...)` | Continue training for one epoch |
| `log_loss`, `deepcopy` | Monitor learning and preserve a fitted checkpoint |
| `model.coefs_`, `model.intercepts_` | Inspect the network's per-layer weights and biases |
| `np.maximum` | Apply the ReLU activation elementwise |

## Tasks

### Task 1: Inspect predictions as images

Select six validation images. Plot each image with its observed and predicted digit in the title. Describe one easy or ambiguous example. Use validation images so you leave the final test assessment unchanged.

In [ ]:
# YOUR CODE: display six validation predictions.

### Task 2: Check the parameter count

Calculate the parameter count by hand from 64 → 32 → 16 → 10, including one bias per non-input unit. Compare it with `parameter_count`. How many extra parameters would changing only the first hidden layer from 32 to 40 add? Use the fitted architecture for this calculation.

In [ ]:
# YOUR CODE: count weights and biases for both architectures.

### Task 3: Reconstruct another prediction

Repeat the manual forward pass for a different validation row. Display its three largest probabilities and check that all ten sum to one. Explain how a close race between the top two classes differs from a clear winner.

In [ ]:
# YOUR CODE: reconstruct and interpret another validation probability vector.